# Q-Mesh: Physical Cryptography — Zero-Interaction Authentication

**Zipminator PQC Platform** | Notebook 08

---

This notebook demonstrates how WiFi CSI sensing from [RuView](https://github.com/MoHoushmand/RuView) creates a new category of security where the physical world IS the authentication system. We visualize:

- 3D Gaussian splatting of WiFi electromagnetic fields
- CSI subcarrier analysis and entropy extraction
- Biometric detection (breathing, heart rate) from WiFi signals
- PUEK (Physical Unclonable Environment Key) enrollment and verification
- Clearance level enforcement (L1-L4)
- EM Canary threat detection
- Full zero-interaction authentication pipeline

> **Disclaimer:** All data in this notebook is synthetically generated to demonstrate the Physical Cryptography concepts. Real CSI data requires RuView ESP32-S3 hardware.

In [ ]:
import sys; sys.path.insert(0, "..")
from _helpers.viz import *

## 1. What is Gaussian Splatting for WiFi CSI?

3D Gaussian splatting (originally from NeRF research) represents a scene as a
collection of 3D Gaussian functions. Each Gaussian has a position, covariance
(shape/orientation), opacity, and color. When rendered, they "splat" onto the
viewing plane to reconstruct a continuous field.

When applied to WiFi Channel State Information (CSI), each Gaussian represents
a region of electromagnetic interaction:

- **Wall reflections**: elongated Gaussians aligned along surfaces where WiFi bounces
- **Furniture scattering**: compact Gaussians at positions where objects diffuse the signal
- **Human body absorption**: large, soft Gaussians where the human body absorbs and re-radiates RF energy

The key insight: the Gaussian splat pattern changes dramatically when people
enter, move, or leave a room. The "shape" of the electromagnetic field is a
fingerprint of the room's physical contents. Since a human body is 60% water
(a strong RF absorber at 2.4/5 GHz), a person's presence creates unmistakable
perturbations in the CSI eigenstructure.

This makes the invisible (WiFi signals bouncing through space) visible and,
more importantly, cryptographically useful.

In [ ]:
# ── 3D Surface: WiFi CSI signal strength across a room grid ──
rng = np.random.RandomState(42)

# Room grid: 6m x 4m, 0.1m resolution
gx = np.linspace(0, 6, 60)
gy = np.linspace(0, 4, 40)
GX, GY = np.meshgrid(gx, gy)

# Base signal: WiFi AP at (1, 2) with inverse-square falloff
ap_x, ap_y = 1.0, 2.0
dist = np.sqrt((GX - ap_x)**2 + (GY - ap_y)**2) + 0.5
signal = 30 / dist

# Wall reflections: boost near walls
wall_boost = 3.0 * np.exp(-GY / 0.5) + 3.0 * np.exp(-(4 - GY) / 0.5)
wall_boost += 2.0 * np.exp(-GX / 0.5) + 2.0 * np.exp(-(6 - GX) / 0.5)
signal += wall_boost

# Furniture scattering: desk at (3, 1.5), bookshelf at (5, 3)
desk_scatter = 8 * np.exp(-((GX - 3)**2 / 0.3 + (GY - 1.5)**2 / 0.2))
shelf_scatter = 6 * np.exp(-((GX - 5)**2 / 0.15 + (GY - 3)**2 / 0.5))
signal += desk_scatter + shelf_scatter

# Human body absorption at (2.5, 2) -- creates a notch
human_absorb = -12 * np.exp(-((GX - 2.5)**2 / 0.15 + (GY - 2)**2 / 0.1))
signal += human_absorb

# Add noise
signal += rng.normal(0, 0.4, signal.shape)

fig = go.Figure(go.Surface(
    x=gx, y=gy, z=signal,
    colorscale=[
        [0.0, "#0a0f1e"], [0.2, "#1e3a5f"], [0.4, "#22d3ee"],
        [0.6, "#a78bfa"], [0.8, "#fb7185"], [1.0, "#fef3c7"],
    ],
    colorbar=dict(
        title=dict(text="Signal (dBm)", font=dict(color="#94a3b8")),
        tickfont=dict(color="#94a3b8"),
    ),
    hovertemplate="X: %{x:.1f}m<br>Y: %{y:.1f}m<br>Signal: %{z:.1f} dBm<extra></extra>",
))

fig.update_layout(
    title="WiFi CSI Signal Strength — 3D Room Map (6m x 4m)",
    scene=dict(
        xaxis=dict(title="X (m)", backgroundcolor="#0f172a", gridcolor="rgba(34,211,238,0.1)"),
        yaxis=dict(title="Y (m)", backgroundcolor="#0f172a", gridcolor="rgba(34,211,238,0.1)"),
        zaxis=dict(title="Signal (dBm)", backgroundcolor="#0f172a", gridcolor="rgba(34,211,238,0.1)"),
        bgcolor="#0a0f1e",
        camera=dict(eye=dict(x=1.6, y=-1.6, z=1.0)),
    ),
    height=600,
    template=ZM_TEMPLATE,
)

# Annotate key features
fig.add_trace(go.Scatter3d(
    x=[ap_x, 3.0, 5.0, 2.5],
    y=[ap_y, 1.5, 3.0, 2.0],
    z=[signal.max() + 2] * 4,
    mode="markers+text",
    marker=dict(size=5, color=[ZM_COLORS["cyan"], ZM_COLORS["amber"], ZM_COLORS["amber"], ZM_COLORS["rose"]]),
    text=["WiFi AP", "Desk", "Bookshelf", "Person"],
    textposition="top center",
    textfont=dict(color="#e2e8f0", size=10),
    hoverinfo="text",
    showlegend=False,
))

fig.show()

## 2. CSI Subcarrier Analysis

Each WiFi frame (802.11n/ac/ax) contains 56 complex subcarrier values in the
Channel State Information. The amplitude and phase of each subcarrier encode
the multipath environment between transmitter and receiver:

$$H(f_k, t) = |H(f_k, t)| \cdot e^{j\phi(f_k, t)}$$

where $H(f_k, t)$ is the channel response at subcarrier frequency $f_k$ and
time $t$.

When a person breathes, their chest wall moves by approximately 1 cm, creating
periodic phase shifts at the breathing frequency (0.2-0.33 Hz, or 12-20
breaths/min). The phase shift on subcarrier $k$ is:

$$\Delta\phi_k(t) = \frac{4\pi \Delta d(t)}{\lambda_k}$$

where $\Delta d(t)$ is the path length change and $\lambda_k$ is the wavelength
at subcarrier $k$. At 5 GHz ($\lambda \approx 6$ cm), a 1 cm chest
displacement produces a phase shift of approximately
$\frac{4\pi \times 0.01}{0.06} \approx 2.1$ radians, which is easily
measurable.

Heart rate creates even smaller motions (~0.1 mm) visible in the CSI
micro-Doppler at 1-2 Hz. These signals require bandpass filtering and
higher-order statistics to extract reliably, but the ESP32-S3's 20 Hz CSI
sampling rate provides sufficient Nyquist margin.

### CSI as a Classical Entropy Source

The phase LSBs of CSI subcarriers carry genuine physical randomness from
electromagnetic scattering, thermal air motion, and multipath interference.
Zipminator's CSI Entropy Harvester applies Von Neumann debiasing to extract
unbiased bits from these measurements.

**This is classical physical randomness, not QRNG.** Quantum random number
generation (used by Zipminator's primary entropy pool via IBM Quantum, QBraid,
and Rigetti backends) derives provably non-deterministic bits from quantum
measurements. CSI entropy is computationally unpredictable but not
information-theoretically secure in the same way.

The two sources are combined via XOR for defense-in-depth: the result is at
least as random as the stronger source. CSI entropy serves as a supplementary
physical fallback, not a replacement for QRNG.

In [ ]:
# ── Line chart: CSI subcarrier amplitude across 56 subcarriers ──
N_SUBCARRIERS = 56
subcarrier_idx = np.arange(N_SUBCARRIERS)
rng = np.random.RandomState(55)

# Empty room: smooth profile with frequency-dependent attenuation
base_amplitude = 25 - 0.15 * subcarrier_idx + 3 * np.sin(subcarrier_idx * 0.3)
empty_room = base_amplitude + rng.normal(0, 0.5, N_SUBCARRIERS)

# Person present: absorption notch in subcarriers 15-35 (body resonance)
person_dip = -6 * np.exp(-((subcarrier_idx - 25)**2) / 50)
person_present = base_amplitude + person_dip + rng.normal(0, 0.8, N_SUBCARRIERS)

# Person breathing: adds periodic modulation visible as variance envelope
breathing_mod = 1.5 * np.sin(subcarrier_idx * 0.2) * np.exp(-((subcarrier_idx - 20)**2) / 80)
person_breathing = person_present + breathing_mod + rng.normal(0, 0.3, N_SUBCARRIERS)

fig = zm_line(
    x=subcarrier_idx,
    y_dict={
        "Empty Room": empty_room,
        "Person Present": person_present,
        "Person Breathing (snapshot)": person_breathing,
    },
    title="CSI Subcarrier Amplitude — Effect of Human Presence",
)
fig.update_layout(
    xaxis_title="Subcarrier Index (0-55)",
    yaxis_title="Amplitude (dB)",
    height=450,
    annotations=[
        dict(
            x=25, y=person_present[25] - 1.5,
            text="Body absorption notch",
            showarrow=True, arrowhead=2,
            font=dict(color=ZM_COLORS["rose"], size=12),
            arrowcolor=ZM_COLORS["rose"],
        ),
    ],
)
fig.show()

## 3. Biometric Profile Detection

Each person has a unique "biometric shape" derived from their interaction with
WiFi signals. The combination of:

- **Respiratory pattern**: rate, depth, regularity (chest wall displacement)
- **Cardiac rhythm**: resting heart rate, HRV (inter-beat interval variation)
- **Postural micro-movements**: sway, fidgeting, tremor

creates a fingerprint that cannot be forged because it comes from the physics
of a specific human body interacting with electromagnetic waves. Unlike a
password (knowledge-based) or a fingerprint scan (possession-based), this
biometric is:

1. **Continuous**: verified every frame, not just at login
2. **Contactless**: no reader, no sensor on the body
3. **Coercion-aware**: stress changes breathing and heart rate, enabling duress detection
4. **Non-replayable**: the biometric includes the room's EM eigenstructure, so a recording from another location fails verification

This is what separates Q-Mesh from traditional biometrics. A fingerprint can
be lifted from a glass. A face can be deepfaked. But you cannot fake the
electromagnetic interaction between a specific human body and a specific room
at a specific moment in time.

In [ ]:
# ── Biometric Profiles: Radar Chart (Scatterpolar) ──
profiles = {
    "Person A (calm)": {"breathing_rate": 14, "heart_rate": 68, "breathing_regularity": 0.92,
        "hrv": 0.85, "movement_amplitude": 0.15, "movement_regularity": 0.88},
    "Person B (active)": {"breathing_rate": 18, "heart_rate": 82, "breathing_regularity": 0.78,
        "hrv": 0.72, "movement_amplitude": 0.45, "movement_regularity": 0.65},
    "Person C (stress)": {"breathing_rate": 22, "heart_rate": 105, "breathing_regularity": 0.45,
        "hrv": 0.38, "movement_amplitude": 0.72, "movement_regularity": 0.35},
}

axes_labels = ["Breathing Rate", "Heart Rate", "Breathing Regularity",
               "HRV", "Movement Amplitude", "Movement Regularity"]
axes_keys = ["breathing_rate", "heart_rate", "breathing_regularity",
             "hrv", "movement_amplitude", "movement_regularity"]
norm_ranges = {"breathing_rate": (10, 30), "heart_rate": (50, 120),
    "breathing_regularity": (0, 1), "hrv": (0, 1),
    "movement_amplitude": (0, 1), "movement_regularity": (0, 1)}

def normalize(value, key):
    lo, hi = norm_ranges[key]
    return (value - lo) / (hi - lo)

# Build values dict for zm_radar
radar_values = {}
for name, profile in profiles.items():
    radar_values[name] = [normalize(profile[k], k) for k in axes_keys]

fig = zm_radar(
    axes_labels,
    radar_values,
    title="WiFi CSI Biometric Profiles: Three Subjects",
)
fig.update_layout(height=550)
fig.show()

In [ ]:
# ── Animated scatter: Person detection via CSI variance over time frames ──
N_FRAMES = 200
FS = 20  # Hz
t = np.arange(N_FRAMES) / FS  # 10 seconds
rng = np.random.RandomState(88)

# Simulate CSI variance across subcarriers over time
base_var = 0.5 + 0.1 * rng.randn(N_FRAMES)

# Person enters at frame 40 (2s), biometric lock at frame 80 (4s)
person_mask = np.zeros(N_FRAMES)
person_mask[40:] = 1.0
breathing_signal = 2.0 * np.sin(2 * np.pi * 0.3 * t) * person_mask  # 18 bpm
heart_signal = 0.3 * np.sin(2 * np.pi * 1.1 * t) * person_mask      # 66 bpm
movement_noise = person_mask * rng.normal(0, 0.8, N_FRAMES)

variance = np.abs(base_var + breathing_signal + heart_signal + movement_noise)

# Labels for coloring
labels = []
for i in range(N_FRAMES):
    if i < 40:
        labels.append("Empty Room")
    elif i < 80:
        labels.append("Person Detected")
    else:
        labels.append("Biometric Lock")

color_map = {
    "Empty Room": ZM_COLORS["cyan"],
    "Person Detected": ZM_COLORS["amber"],
    "Biometric Lock": ZM_COLORS["emerald"],
}
colors = [color_map[l] for l in labels]

# Build animated frames (every 5 data points = 0.25s per frame)
frame_step = 5
anim_frames = []
for end in range(frame_step, N_FRAMES + 1, frame_step):
    anim_frames.append(go.Frame(
        data=[go.Scatter(
            x=t[:end], y=variance[:end],
            mode="markers",
            marker=dict(size=6, color=colors[:end], opacity=0.8,
                        line=dict(width=0.5, color="rgba(34,211,238,0.3)")),
            hovertemplate="t=%{x:.2f}s<br>Variance=%{y:.2f}<extra></extra>",
        )],
        name=f"t={t[end-1]:.1f}s",
    ))

fig = go.Figure(
    data=[go.Scatter(
        x=t[:frame_step], y=variance[:frame_step],
        mode="markers",
        marker=dict(size=6, color=colors[:frame_step], opacity=0.8,
                    line=dict(width=0.5, color="rgba(34,211,238,0.3)")),
    )],
    frames=anim_frames,
)

fig.update_layout(
    title="Person Detection via CSI Variance Over Time",
    xaxis_title="Time (s)",
    yaxis_title="CSI Variance (a.u.)",
    xaxis=dict(range=[0, 10.5]),
    yaxis=dict(range=[0, variance.max() * 1.3]),
    height=450,
    template=ZM_TEMPLATE,
    updatemenus=[dict(
        type="buttons", showactive=False, x=0.05, y=1.12,
        buttons=[
            dict(label="▶ Play", method="animate",
                 args=[None, {"frame": {"duration": 120}, "fromcurrent": True}]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], {"frame": {"duration": 0}, "mode": "immediate"}]),
        ],
    )],
    sliders=[dict(
        active=0, yanchor="top", y=-0.1, x=0.05, len=0.9,
        steps=[
            dict(args=[[f.name], {"frame": {"duration": 120}, "mode": "immediate"}],
                 label=f.name, method="animate")
            for f in anim_frames
        ],
    )],
    annotations=[
        dict(x=1.0, y=0.5, text="Empty Room",
             showarrow=False, font=dict(color=ZM_COLORS["cyan"], size=11)),
        dict(x=3.0, y=variance.max() * 1.15, text="Person enters",
             showarrow=True, arrowhead=2, ax=0, ay=-30,
             font=dict(color=ZM_COLORS["amber"], size=11),
             arrowcolor=ZM_COLORS["amber"]),
        dict(x=4.0, y=variance.max() * 1.05, text="Biometric lock acquired",
             showarrow=True, arrowhead=2, ax=40, ay=-25,
             font=dict(color=ZM_COLORS["emerald"], size=11),
             arrowcolor=ZM_COLORS["emerald"]),
    ],
)

fig.show()

## 4. PUEK: The Room as Your Key

A **Physical Unclonable Environment Key** (PUEK) exploits the fact that every
room has a unique electromagnetic signature. The CSI covariance matrix:

$$\mathbf{R} = \frac{1}{T} \sum_{t=1}^{T} \mathbf{h}(t) \mathbf{h}(t)^H$$

where $\mathbf{h}(t) \in \mathbb{C}^{56}$ is the CSI vector at time $t$, encodes
the multipath structure of the room. The eigenvalue decomposition
$\mathbf{R} = \mathbf{U} \mathbf{\Lambda} \mathbf{U}^H$ yields eigenvalues
$\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_{56}$ that form the room's
fingerprint.

Properties that make PUEK useful for security:

1. **Stability**: The top eigenvalues are dominated by walls and large surfaces,
   which do not move. They are stable across hours and days.
2. **Uniqueness**: No two rooms have the same geometry, materials, and furniture
   arrangement. The eigenvalue spectrum is as unique as a fingerprint.
3. **Unclonability**: To spoof a PUEK, an attacker would need to physically
   reconstruct the room's geometry and materials to sub-centimeter precision.
4. **Liveness**: The eigenvalues shift when people enter/exit, providing built-in
   liveness detection.

During **enrollment**, the system captures the empty-room eigenvalue spectrum and
stores it as the reference PUEK. During **verification**, the live eigenvalue
spectrum is compared against the stored reference using cosine similarity.

In [ ]:
# ── PUEK eigenvalue spectra comparison ──
N_EIGENVALUES = 10
eig_idx = np.arange(1, N_EIGENVALUES + 1)

enrolled = np.array([45.2, 28.7, 18.3, 12.1, 8.5, 5.9, 3.8, 2.4, 1.5, 0.9])

rng = np.random.RandomState(99)
same_room = enrolled + rng.normal(0, 0.6, N_EIGENVALUES)
same_room = np.maximum(same_room, 0.1)

diff_room = np.array([52.1, 22.4, 15.8, 14.2, 9.8, 7.1, 6.3, 4.0, 3.2, 2.8])

def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

sim_same = cosine_sim(enrolled, same_room)
sim_diff = cosine_sim(enrolled, diff_room)

fig = zm_grouped_bar(
    eig_idx.tolist(),
    {
        "Enrolled Reference": enrolled.tolist(),
        f"Same Room (sim={sim_same:.4f})": same_room.tolist(),
        f"Different Room (sim={sim_diff:.4f})": diff_room.tolist(),
    },
    title="PUEK Eigenvalue Spectra \u2014 Room Fingerprint Verification",
)

fig.update_layout(
    xaxis_title="Eigenvalue Rank",
    yaxis_title="Eigenvalue Magnitude",
    height=450,
    annotations=[
        dict(x=0.5, y=1.08, xref="paper", yref="paper",
             text=f"Threshold: 0.95 | Same room: {sim_same:.4f} PASS | Different room: {sim_diff:.4f} FAIL",
             showarrow=False, font=dict(color="#94a3b8", size=12)),
    ],
)

fig.show()

## 5. Clearance Level Enforcement

Q-Mesh implements four clearance levels, each layering additional cryptographic
primitives:

| Level | Name | Requirements | Use Case |
|-------|------|-------------|----------|
| L1 | Basic | PUEK room match | Office access, meeting rooms |
| L2 | Standard | L1 + biometric profile match | Workstations, file access |
| L3 | High | L2 + vital sign validation + duress check | Server rooms, financial systems |
| L4 | Critical | L3 + EM Canary sweep + topology lock | SCIF, nuclear, command centers |

Each level is a strict superset of the previous one. The system never
downgrades; if a higher-level check fails, the session is terminated even if
lower-level checks passed. This follows the principle of **fail-closed
security**: ambiguity is resolved in favor of denial.

In [ ]:
# ── Bar chart: Mesh node key rotation intervals vs security level ──
levels = ["L1: Basic", "L2: Standard", "L3: High", "L4: Critical"]
rotation_intervals = [3600, 900, 60, 5]  # seconds
rotation_labels = ["1 hour", "15 min", "1 min", "5 sec"]
n_checks = [1, 2, 4, 6]  # number of crypto checks per level
colors = [ZM_COLORS["cyan"], ZM_COLORS["emerald"], ZM_COLORS["amber"], ZM_COLORS["rose"]]

fig = zm_subplots(1, 2, titles=["Key Rotation Interval", "Cryptographic Checks per Level"])

fig.add_trace(go.Bar(
    x=levels, y=rotation_intervals,
    marker_color=colors,
    text=rotation_labels,
    textposition="auto",
    textfont=dict(color="#f8fafc", size=12),
    hovertemplate="%{x}<br>Rotation: %{text}<br>(%{y}s)<extra></extra>",
    showlegend=False,
), row=1, col=1)

check_names = ["PUEK", "Biometric", "Vital Signs", "Duress", "EM Canary", "Topology"]
check_matrix = [
    [1, 0, 0, 0, 0, 0],  # L1
    [1, 1, 0, 0, 0, 0],  # L2
    [1, 1, 1, 1, 0, 0],  # L3
    [1, 1, 1, 1, 1, 1],  # L4
]
check_colors = [
    ZM_COLORS["cyan"], ZM_COLORS["violet"], ZM_COLORS["emerald"],
    ZM_COLORS["amber"], ZM_COLORS["rose"], ZM_COLORS["blue"],
]

for ci, cname in enumerate(check_names):
    vals = [check_matrix[li][ci] for li in range(4)]
    fig.add_trace(go.Bar(
        x=levels, y=vals, name=cname,
        marker_color=check_colors[ci],
        showlegend=True,
    ), row=1, col=2)

fig.update_layout(
    height=450,
    barmode="stack",
    yaxis_title="Rotation Interval (seconds)",
    yaxis_type="log",
    yaxis2_title="Active Checks",
)

fig.show()

## 6. EM Canary: Physical Intrusion Detection

The EM Canary system continuously monitors the room's eigenstructure baseline
against live CSI readings. The deviation metric is computed as:

$$\delta(t) = 1 - \text{sim}(\boldsymbol{\lambda}_{\text{baseline}}, \boldsymbol{\lambda}(t))$$

where $\text{sim}$ is cosine similarity and $\boldsymbol{\lambda}(t)$ is the current
eigenvalue vector.

Three threat zones:

- **Green** ($\delta < 5\%$): Normal operation. Small fluctuations from HVAC,
  lighting changes, minor vibrations.
- **Amber** ($5\% \leq \delta < 20\%$): Known perturbation. A person entered,
  a door opened, furniture moved. The system attempts to re-identify the source
  and re-baseline.
- **Red** ($\delta \geq 20\%$): Unknown perturbation. Could be a hidden camera,
  an RF transmitter (bug), an unauthorized person in an adjacent room, or RF
  jamming. Keys are rotated immediately and the security team is alerted.

The EM Canary catches threats that software-only systems cannot see. A hidden
camera emits RF from its wireless transmitter. A person in an adjacent
thin-walled room perturbs the EM field through the wall. An RF jammer creates
massive deviation. None of these are visible to a firewall or IDS.

In [ ]:
# ── EM Canary timeline with threat zones ──
T_CANARY = 60
FS_CANARY = 10
N_CANARY = T_CANARY * FS_CANARY
t_canary = np.linspace(0, T_CANARY, N_CANARY)

rng = np.random.RandomState(77)

# Build deviation timeline
deviation = np.ones(N_CANARY) * 2.5 + rng.normal(0, 0.8, N_CANARY)

# Event 1: Person enters (t=20-25s)
mask1 = (t_canary >= 20) & (t_canary < 25)
deviation[mask1] = 15 + rng.normal(0, 2, mask1.sum())

# Event 2: Person identified, re-baselined (t=25-30s)
mask2 = (t_canary >= 25) & (t_canary < 30)
deviation[mask2] = np.linspace(12, 3, mask2.sum()) + rng.normal(0, 0.5, mask2.sum())

# Event 3: RF bug detected (t=42-48s)
mask3 = (t_canary >= 42) & (t_canary < 48)
deviation[mask3] = 25 + rng.normal(0, 3, mask3.sum())

# Event 4: Bug removed, decay (t=48-54s)
mask4 = (t_canary >= 48) & (t_canary < 54)
deviation[mask4] = np.linspace(22, 4, mask4.sum()) + rng.normal(0, 1, mask4.sum())

deviation = np.clip(deviation, 0, 35)

# Color by zone
zone_colors = []
for d in deviation:
    if d < 5:
        zone_colors.append(ZM_COLORS["emerald"])
    elif d < 20:
        zone_colors.append(ZM_COLORS["amber"])
    else:
        zone_colors.append(ZM_COLORS["rose"])

fig = go.Figure()

# Threat zone backgrounds
fig.add_hrect(y0=0, y1=5, fillcolor=ZM_COLORS["emerald"], opacity=0.08,
              line_width=0, annotation_text="GREEN", annotation_position="top left",
              annotation=dict(font=dict(color=ZM_COLORS["emerald"], size=10)))
fig.add_hrect(y0=5, y1=20, fillcolor=ZM_COLORS["amber"], opacity=0.06,
              line_width=0, annotation_text="AMBER", annotation_position="top left",
              annotation=dict(font=dict(color=ZM_COLORS["amber"], size=10)))
fig.add_hrect(y0=20, y1=35, fillcolor=ZM_COLORS["rose"], opacity=0.06,
              line_width=0, annotation_text="RED", annotation_position="top left",
              annotation=dict(font=dict(color=ZM_COLORS["rose"], size=10)))

fig.add_trace(go.Scatter(
    x=t_canary, y=deviation, mode="markers+lines",
    marker=dict(size=3, color=zone_colors),
    line=dict(color="rgba(148,163,184,0.4)", width=1),
    hovertemplate="t=%{x:.1f}s<br>Deviation=%{y:.1f}%<extra></extra>",
    showlegend=False,
))

fig.update_layout(
    title="EM Canary — 60-Second Threat Timeline",
    xaxis_title="Time (seconds)",
    yaxis_title="Eigenstructure Deviation (%)",
    yaxis=dict(range=[0, 35]),
    height=450,
    template=ZM_TEMPLATE,
    annotations=[
        dict(x=22.5, y=17, text="Person enters", showarrow=True, arrowhead=2,
             ax=30, ay=-25, font=dict(color=ZM_COLORS["amber"], size=11),
             arrowcolor=ZM_COLORS["amber"]),
        dict(x=27, y=7, text="Re-baselined", showarrow=True, arrowhead=2,
             ax=30, ay=-20, font=dict(color=ZM_COLORS["emerald"], size=11),
             arrowcolor=ZM_COLORS["emerald"]),
        dict(x=45, y=28, text="RF BUG DETECTED", showarrow=True, arrowhead=2,
             ax=0, ay=-30, font=dict(color=ZM_COLORS["rose"], size=12),
             arrowcolor=ZM_COLORS["rose"]),
    ],
)

fig.show()

## 7. Room Occupancy Detection

By monitoring CSI variance across the mesh nodes, Q-Mesh can build a
room-by-room occupancy map. Each node pair (TX-RX link) measures the local
electromagnetic environment; high variance on a link indicates human presence
in the path between those nodes.

In [ ]:
# ── Heatmap: Room occupancy detection confidence ──
rooms = ["Lobby", "Office A", "Office B", "Server Room", "Lab", "Break Room", "SCIF", "Conference"]
time_slots = ["08:00", "09:00", "10:00", "11:00", "12:00", "13:00", "14:00", "15:00", "16:00", "17:00"]

rng = np.random.RandomState(33)

# Occupancy confidence (0-100%): simulate realistic office patterns
occupancy = np.zeros((len(rooms), len(time_slots)))

# Lobby: busy morning, lunch, evening
occupancy[0] = [85, 60, 30, 25, 70, 65, 30, 25, 45, 80]
# Office A: occupied during work hours
occupancy[1] = [20, 92, 95, 90, 40, 88, 93, 91, 85, 30]
# Office B: similar but different person
occupancy[2] = [15, 45, 88, 91, 35, 42, 90, 87, 80, 25]
# Server Room: rarely occupied
occupancy[3] = [0, 0, 0, 15, 0, 0, 0, 20, 0, 0]
# Lab: intermittent
occupancy[4] = [0, 30, 75, 80, 25, 65, 85, 70, 45, 10]
# Break Room: lunch peak
occupancy[5] = [40, 20, 15, 30, 90, 85, 20, 15, 25, 35]
# SCIF: rare, controlled access
occupancy[6] = [0, 0, 0, 0, 0, 0, 45, 50, 0, 0]
# Conference: meetings
occupancy[7] = [0, 70, 0, 85, 0, 0, 75, 0, 0, 0]

# Add noise
occupancy += rng.normal(0, 3, occupancy.shape)
occupancy = np.clip(occupancy, 0, 100)

fig = zm_heatmap(
    z=occupancy,
    x=time_slots,
    y=rooms,
    title="Q-Mesh Room Occupancy Detection Confidence (%)",
    colorscale=[
        [0.0, "#0a0f1e"], [0.15, "#1e293b"], [0.3, "#164e63"],
        [0.5, "#22d3ee"], [0.7, "#a78bfa"], [0.85, "#fb7185"], [1.0, "#fef3c7"],
    ],
)
fig.update_layout(
    xaxis_title="Time of Day",
    yaxis_title="Room",
    height=500,
)
fig.update_traces(
    hovertemplate="%{y} @ %{x}<br>Confidence: %{z:.0f}%<extra></extra>",
    text=np.round(occupancy).astype(int),
    texttemplate="%{text}",
    textfont=dict(size=10, color="#e2e8f0"),
)

fig.show()

## 8. The Complete Zero-Interaction Auth Pipeline

From the user's perspective, they simply walk into the room. There is no login
screen, no badge tap, no fingerprint reader. Behind the scenes, six
cryptographic primitives work in concert:

1. **CSI Entropy**: Raw phase data from WiFi subcarriers seeds the session key
   material. This entropy is information-theoretically unpredictable because it
   depends on the exact multipath environment at the exact moment of capture.

2. **PUEK Verification**: The room's eigenvalue spectrum is compared against the
   enrolled reference. This confirms the session is occurring in an authorized
   physical location.

3. **Vital Authentication**: Breathing rate, heart rate, and micro-movement
   patterns are extracted from CSI and matched against the enrolled user's
   biometric profile. Continuous, not one-shot.

4. **EM Canary**: The room's EM field is continuously monitored for anomalies
   that indicate physical intrusion, surveillance devices, or RF jamming.

5. **Topology Authentication**: The network graph of all RuView nodes is locked.
   Any node that disappears, appears, or changes position triggers
   re-authentication.

6. **Spatiotemporal Signatures**: Every authentication event is signed with a
   timestamp and location hash, providing legal non-repudiation for audit
   trails (DORA Art. 7 compliance).

No passwords. No badges. No biometric scanners. The physics does the work.

In [ ]:
# ── Pipeline timeline + confidence build-up ──
pipeline_stages = [
    "Person Enters", "CSI Captured", "Entropy Extracted",
    "PUEK Verified", "Biometric Matched", "Vitals Validated",
    "EM Canary Clear", "Topology Locked", "SESSION ACTIVE",
]
stage_times = [0.0, 0.1, 0.5, 0.8, 1.5, 2.0, 2.2, 2.4, 2.5]  # seconds
stage_confidence = [0, 60, 72, 85, 91, 95, 97, 99, 99.7]  # cumulative %
stage_colors = [
    "#475569", ZM_COLORS["blue"], ZM_COLORS["cyan"],
    ZM_COLORS["violet"], ZM_COLORS["emerald"], ZM_COLORS["teal"],
    ZM_COLORS["amber"], ZM_COLORS["indigo"], ZM_COLORS["rose"],
]

fig = zm_subplots(
    2, 1, titles=["Zero-Interaction Auth Pipeline", "Cumulative Confidence"],
    row_heights=[0.5, 0.5], vertical_spacing=0.15,
)

# Top: timeline bar chart
fig.add_trace(go.Bar(
    y=pipeline_stages, x=[stage_times[i+1] - stage_times[i] if i < len(stage_times)-1 else 0.1
                          for i in range(len(pipeline_stages))],
    orientation="h",
    marker_color=stage_colors,
    text=[f"{t:.1f}s" for t in stage_times],
    textposition="auto",
    textfont=dict(color="#f8fafc", size=11),
    hovertemplate="%{y}<br>Start: %{text}<extra></extra>",
    showlegend=False,
), row=1, col=1)

# Bottom: confidence line
fig.add_trace(go.Scatter(
    x=stage_times, y=stage_confidence,
    mode="lines+markers+text",
    line=dict(color=ZM_COLORS["cyan"], width=3),
    marker=dict(size=10, color=stage_colors, line=dict(width=2, color="#0a0f1e")),
    text=[f"{c}%" for c in stage_confidence],
    textposition="top center",
    textfont=dict(color="#e2e8f0", size=10),
    hovertemplate="t=%{x:.1f}s<br>Confidence: %{y:.1f}%<extra></extra>",
    showlegend=False,
), row=2, col=1)

# Threshold line
fig.add_hline(y=95, line_dash="dash", line_color=ZM_COLORS["emerald"],
              annotation_text="L3 threshold (95%)",
              annotation_font_color=ZM_COLORS["emerald"],
              row=2, col=1)

fig.update_layout(
    height=700,
    xaxis_title="Duration (seconds)",
    xaxis2_title="Time (seconds)",
    yaxis2_title="Confidence (%)",
)

fig.show()

## Summary

---

### Key Takeaways

**1. WiFi CSI is an unclonable security substrate.** The electromagnetic
eigenstructure of a room is determined by geometry, materials, and contents at
sub-wavelength precision. You cannot fake it without physically reconstructing
the entire environment.

**2. Biometrics from WiFi signals are contactless, continuous, and
coercion-aware.** Breathing rate and heart rate extracted from CSI phase shifts
provide identity verification that works through walls, requires no hardware on
the person, and detects stress/duress automatically.

**3. Clearance levels L1-L4 layer primitives for progressive security.** Each
level adds cryptographic checks. L1 verifies the room. L2 adds biometric
identity. L3 validates vital signs and detects coercion. L4 sweeps for
physical surveillance devices and locks the network topology.

**4. The system requires zero user interaction.** Authentication happens by
physics. Walk into the room, the room authenticates you. No passwords, no
badges, no biometric scanners. The ESP32-S3 nodes do all the sensing at 20 Hz.

**5. EM Canary detects physical intrusions that software-only systems cannot
see.** Hidden cameras, RF bugs, unauthorized presence in adjacent rooms, and RF
jamming all create measurable perturbations in the CSI eigenstructure. Firewalls
and IDS are blind to these threats; Q-Mesh is not.

---

> All data in this notebook is synthetically generated. Production deployment
> requires [RuView ESP32-S3 hardware](https://github.com/MoHoushmand/RuView)
> for real CSI data acquisition.

**Next notebook:** [09 - Deployment & Operations](09_deployment_ops.ipynb)